In [1]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.8 MB/s eta 0:00:00


In [5]:
import os
from groq import Groq

# Set your API key here directly
os.environ["GROQ_API_KEY"] = "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"

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in various aspects of natural language processing (NLP) and artificial intelligence (AI). The importance of fast language models can be understood from the following perspectives:

1. **Efficient Processing**: Fast language models can process large amounts of text data quickly, enabling real-time applications such as chatbots, virtual assistants, and language translation software. This is particularly important in applications where responsiveness is critical, such as customer service or emergency response systems.
2. **Scalability**: Fast language models can handle a large volume of requests, making them suitable for large-scale applications such as social media platforms, online forums, and content moderation systems. This scalability is essential for supporting a growing user base and handling increased traffic.
3. **Low Latency**: Fast language models can respond to user input quickly, reducing latency and improving the overall user experience. This

In [6]:
class Agent:
    def __init__(self, client: Groq, system: str = "") -> None:
        self.client = client
        self.system = system
        self.messages: list = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile", messages=self.messages
        )
        return completion.choices[0].message.content

In [7]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this:

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()


def calculate(operation: str) -> float:
    return eval(operation)


def get_planet_mass(planet) -> float:
    match planet.lower():
        case "earth":
            return 5.972e24
        case "jupiter":
            return 1.898e27
        case "mars":
            return 6.39e23
        case "mercury":
            return 3.285e23
        case "neptune":
            return 1.024e26
        case "saturn":
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "venus":
            return 4.867e24
        case _:
            return 0.0

print("System prompt and tools ready!")

System prompt and tools ready!


In [8]:
neil_tyson = Agent(client=client, system=system_prompt)

In [9]:
result = neil_tyson("What is the mass of Mercury times 5?")
print(result)

Thought: I need to find the mass of Mercury, then multiply it by 5 to get the final answer.
Action: get_planet_mass: Mercury
PAUSE


In [10]:
result = neil_tyson()
print(result)

In [11]:
result = get_planet_mass("mercury")
print(result)

3.285e+23


In [12]:
next_prompt = "Observation: {}".format(result)
print(next_prompt)


Observation: 3.285e+23


In [13]:
result = neil_tyson(next_prompt)
print(result)

Thought: I now have the mass of Mercury, which is 3.285e+23 kg. To find the mass of Mercury times 5, I need to multiply this value by 5.
Action: calculate: 3.285e+23 * 5
PAUSE


In [14]:
result = calculate("3.285e+23 * 5")
print(result)

1.6425e+24


In [15]:
next_prompt = "Observation: {}".format(result)
result = neil_tyson(next_prompt)
print(result)

Thought: I have now calculated the mass of Mercury times 5, which is 1.6425e+24 kg. This is the final answer to the question.
Answer: The mass of Mercury times 5 is 1.6425e+24.


In [16]:
for msg in neil_tyson.messages:
    print(msg['content'])
    print("---")

You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE 

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this: 

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 